# Prebuilt Agents with `create_agent`

This notebook builds the same kind of tool-using agent as `15_Tools/Tooling.ipynb`,
but with LangChain's prebuilt `create_agent` function instead of a hand-built
`StateGraph` + `ToolNode` + `tools_condition` loop. `create_agent` still runs on
LangGraph underneath; it just builds the graph for you. The notebook then adds four
things on top of the basic agent: structured output, multi-turn memory, runtime
context passed into a tool, and middleware (a hook that runs around the agent's
steps, demonstrated here with human-in-the-loop approval).

In [ ]:
from dataclasses import dataclass
from typing import Literal

from dotenv import load_dotenv
from pydantic import BaseModel

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.tools import ToolRuntime
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

In [ ]:
load_dotenv()

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)

## Part 1: A Basic Agent With a Tool

First, a normal tool: a plain Python function decorated with `@tool`, exactly like
in `15_Tools/Tooling.ipynb`.

In [ ]:
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    fake_weather = {
        "paris": "18C and cloudy",
        "tokyo": "24C and sunny",
        "cairo": "33C and clear",
    }
    return fake_weather.get(city.lower(), "20C and mild")

`create_agent` takes the model, the list of tools, and a system prompt, and returns
a compiled LangGraph graph. That graph is the same reason/act/observe loop from
`15_Tools/Tooling.ipynb`: a chat node that can call the model, a `ToolNode`-style
step that runs any requested tool, and a routing rule that sends control back to the
model after a tool runs. `create_agent` builds all of that for you instead of you
writing `StateGraph`, `ToolNode`, and `tools_condition` by hand.

In [ ]:
agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="You are a helpful assistant. Use tools when you need current data.",
)

The agent is invoked the same way as any LangGraph app: with a `messages` list in
the input dictionary.

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "What is the weather in Tokyo?"}]})
for m in result["messages"]:
    print(type(m).__name__, "-", getattr(m, "content", None) or m.tool_calls)

## Part 2: Structured Output

By default, an agent's final answer is a plain-text `AIMessage`. Passing
`response_format=SomePydanticModel` makes the agent also produce a validated object
of that shape, available at `result["structured_response"]`. Internally, the agent
adds one more forced tool call at the end of the turn: a "tool" whose schema is the
pydantic model, so the model's last action is producing structured data instead of
free text.

In [ ]:
class WeatherReport(BaseModel):
    city: str
    condition: str

This cell uses `gemini-2.5-flash` instead of `gemini-2.5-flash-lite`, and that switch
matters here. Side-by-side testing while building this notebook showed that when an
agent has both a regular tool (`get_weather`) and `response_format` set,
`gemini-2.5-flash-lite` calls `get_weather`, reads the result, and then just answers
in plain text — it never makes the forced structured-output tool call, so
`result["structured_response"]` stays `None`. The identical code with
`gemini-2.5-flash` calls `get_weather` first and then correctly calls the
`WeatherReport` tool as its last step. This is a real difference between the two
models, not a bug in the code below.

In [ ]:
flash_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

structured_agent = create_agent(
    model=flash_llm,
    tools=[get_weather],
    system_prompt="You are a helpful assistant. Use tools when you need current data.",
    response_format=WeatherReport,
)

result = structured_agent.invoke({"messages": [{"role": "user", "content": "What is the weather in Cairo?"}]})
print(result["structured_response"])

## Part 3: Memory Across Turns

Passing `checkpointer=InMemorySaver()` gives the agent memory: LangGraph saves the
state after every step, keyed by a `thread_id`. Invoking the agent again with the
same `thread_id` continues the same conversation instead of starting a new one. This
is the same checkpointer idea used in `09_Persistence`, just wired into a prebuilt
agent instead of a hand-built graph.

In [ ]:
checkpointer = InMemorySaver()
memory_agent = create_agent(model=llm, tools=[get_weather], checkpointer=checkpointer)

config = {"configurable": {"thread_id": "weather-chat-1"}}

result = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "What is the weather in Paris?"}]},
    config=config,
)
print(result["messages"][-1].content)

In [ ]:
result = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "Which city did I just ask about?"}]},
    config=config,
)
print(result["messages"][-1].content)

## Part 4: Runtime Context

Sometimes a tool needs information that should not come from the conversation, like
the current user's ID or preferences. `context_schema` declares the shape of that
information, and `agent.invoke(..., context=...)` passes an instance of it in. A
tool reads it by adding a parameter named `runtime` typed as `ToolRuntime`; LangGraph
injects a `ToolRuntime` object into that parameter automatically, and `runtime.context`
holds the value passed to `context=`.

In [ ]:
@dataclass
class UserContext:
    user_name: str


@tool
def greet_user(runtime: ToolRuntime[UserContext]) -> str:
    """Greet the current user by name."""
    return f"Hello, {runtime.context.user_name}!"

In [ ]:
context_agent = create_agent(model=llm, tools=[greet_user], context_schema=UserContext)

result = context_agent.invoke(
    {"messages": [{"role": "user", "content": "Greet me please."}]},
    context=UserContext(user_name="Chinmoy"),
)
print(result["messages"][-1].content)

## Part 5: Middleware

Middleware is code that runs before or after the model or a tool call, without
changing how the tool or the model itself works. It is how `create_agent` adds
optional behavior (approval steps, retries, summarizing old messages, guardrails)
without every project having to rebuild the agent loop from scratch.

`HumanInTheLoopMiddleware` pauses the agent right before a chosen tool runs and waits
for a human decision. `interrupt_on={"send_email": True}` marks `send_email` as a
tool that always requires approval.

In [ ]:
@tool
def send_email(to: str, body: str) -> str:
    """Send an email to someone."""
    return f"Email sent to {to}: {body}"

In [ ]:
hitl_checkpointer = InMemorySaver()
email_agent = create_agent(
    model=llm,
    tools=[send_email],
    system_prompt="You help the user send emails.",
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"send_email": True})],
    checkpointer=hitl_checkpointer,
)

A checkpointer is required for HITL middleware: the graph has to be paused and later
resumed, which only works if its state was saved. Invoking the agent now stops before
`send_email` runs and returns an `__interrupt__` entry describing the pending call
instead of a final answer.

In [ ]:
config = {"configurable": {"thread_id": "email-1"}}
result = email_agent.invoke(
    {"messages": [{"role": "user", "content": "Send an email to bob@example.com saying hi"}]},
    config=config,
)
print("paused:", "__interrupt__" in result)
print(result["__interrupt__"][0].value)

To continue, invoke the agent again with `Command(resume=...)` instead of a new
message. The exact payload the middleware expects is a dictionary with a
`"decisions"` list, one decision per pending tool call. Each decision has a `"type"`
of `"approve"`, `"edit"`, `"reject"`, or `"respond"` (this is what `review_configs`
in the interrupt payload above lists as `allowed_decisions`; the LangChain docs use
the word "accept" in places, but the middleware code itself only accepts
`"approve"`).

In [ ]:
result = email_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config,
)
for m in result["messages"]:
    print(type(m).__name__, "-", getattr(m, "content", None) or m.tool_calls)

## Summary

`create_agent` builds the same LLM-decides / tool-executes / LLM-answers loop as the
hand-built graph in `15_Tools`, as a single function call. On top of the basic loop
it adds, through plain keyword arguments: `response_format` for structured output,
`checkpointer` for multi-turn memory, `context_schema` for passing runtime data into
tools, and `middleware` for cross-cutting behavior like human approval. Another
built-in middleware worth knowing about is `SummarizationMiddleware`, which
compresses old messages once the conversation gets long, so the model does not run
out of context window; it is not run in this notebook, but it plugs into the same
`middleware=[...]` list.